In [0]:
%pip install polars
!pip install fastexcel

In [0]:
import pprint

## Labels

In [0]:
import pandas as pd

df_excel = pd.read_excel('/Workspace/Users/hakeemfujah@tfl.gov.uk/Corporate Archives/Autoclassification Scheme/Wholly Amalgamated Series Draft 07.01.2025.xlsx')

In [0]:
display(df_excel.tail())

In [0]:
cols = ['Ref No'] + [f'Level {i}' for i in range(1, len(df_excel.columns))]
df_excel.columns = cols

In [0]:
len(df_excel)

In [0]:
df_excel.columns

In [0]:
import pandas as pd

ref_label_pairs = []

for i in range(df_excel.shape[1] - 1):
    ref_col = df_excel.iloc[:, i]
    label_col = df_excel.iloc[:, i + 1]
    for ref, label in zip(ref_col, label_col):
        if isinstance(ref, str) and ref.startswith('LT'):
            ref_label_pairs.append({'Reference': ref.strip(), 'Name': str(label).strip() if label is not None else ""})

folder_mapping = pd.DataFrame(ref_label_pairs)
display(folder_mapping)

In [0]:
folder_mapping['Name'] = folder_mapping['Name'].astype(str)

In [0]:
folder_mapping['Name'].info()

In [0]:
folder_mapping[folder_mapping['Reference'] == "LT000260/001/003/06"]

In [0]:
folder_mapping[folder_mapping['Name'] == "nan"]

In [0]:
folder_mapping = folder_mapping[folder_mapping['Name'] != 'nan']
display(folder_mapping)

In [0]:
# Function to extract reference-label pairs column-wise
def extract_columnwise_pairs(df):
    column_pairs = []

    # Iterate over columns in pairs (ref_no, label)
    for i in range(0, df.shape[1] - 1):
        ref_col = df.iloc[:, i]  # Fix: Use .iloc instead of square brackets for column indexing
        label_col = df.iloc[:, i + 1]  # Fix: Use .iloc instead of square brackets for column indexing

        ref_label_pairs = []
        for ref, label in zip(ref_col, label_col):
            ref_str = str(ref).strip() if ref is not None else ""
            label_str = str(label).strip() if label is not None else ""
            if ref_str and label_str:
                ref_label_pairs.append((ref_str, label_str))

        # Create a DataFrame for this column pair
        if ref_label_pairs:
            pair_df = pd.DataFrame({
                "Reference": [pair[0] for pair in ref_label_pairs],
                "Label": [pair[1] for pair in ref_label_pairs]
            })
            column_pairs.append(pair_df)

    return column_pairs

# Apply the function to df_excel
columnwise_hierarchies = extract_columnwise_pairs(df_excel)

# Display the number of extracted column-wise hierarchies
print(f"Extracted {len(columnwise_hierarchies)} column-wise reference-label mappings.")

In [0]:
pprint.pprint(columnwise_hierarchies)

In [0]:
# Display each level's DataFrame (up to 7) in the notebook, assigning each to its own variable
for level, df in enumerate(columnwise_hierarchies[:7], start=1):
    vars()[f"df_level_{level}"] = df
    display(df)

In [0]:
len(columnwise_hierarchies)

In [0]:
import pandas as pd

# Combine all levels into a single DataFrame showing the full hierarchy path

# List to collect hierarchy rows as dictionaries
hierarchy_rows = []
# Determine the number of levels in the hierarchy
max_level = len(columnwise_hierarchies)

# Convert Polars DataFrames to Pandas DataFrames and annotate each with its level number
for level_idx in range(max_level):
    columnwise_hierarchies[level_idx]["Level"] = level_idx + 1

# Recursive function to build hierarchy paths and collect them as rows
def build_hierarchy(path, labels, level):
    if level >= max_level:
        row_dict = {}
        path_parts = path.split('/')
        for i in range(len(labels)):
            row_dict[f"Level_{i+1}_Ref"] = path_parts[i] if i < len(path_parts) else None
            row_dict[f"Level_{i+1}_Name"] = labels[i]
        row_dict['folder_ref'] = path
        hierarchy_rows.append(row_dict)
        return
    current_df = columnwise_hierarchies[level]
    matches = current_df[current_df["Reference"].str.startswith(path + "/")]
    if matches.empty:
        row_dict = {}
        path_parts = path.split('/')
        for i in range(len(labels)):
            row_dict[f"Level_{i+1}_Ref"] = path_parts[i] if i < len(path_parts) else None
            row_dict[f"Level_{i+1}_Name"] = labels[i]
        row_dict['folder_ref'] = path
        hierarchy_rows.append(row_dict)
        return
    for _, row in matches.iterrows():
        new_path = row['Reference']
        new_label = row['Label']
        build_hierarchy(new_path, labels + [new_label], level + 1)

# Start building the hierarchy from the top-level nodes
for _, row in columnwise_hierarchies[0].iterrows():
    base_ref = row['Reference']
    base_label = row['Label']
    build_hierarchy(base_ref, [base_label], 1)

# Create a DataFrame from the collected hierarchy rows and display it
hierarchy_df = pd.DataFrame(hierarchy_rows)
cols = [col for col in hierarchy_df.columns if col != 'folder_ref'] + ['folder_ref']
hierarchy_df = hierarchy_df[cols]
display(hierarchy_df)

In [0]:
hierarchy_df[hierarchy_df['Level_1_Ref'] == 'LT000257']

In [0]:
hierarchy_df.columns

In [0]:
hierarchy_df[hierarchy_df['Level_3_Name'] == 'Board, Committee, and Panel Members Details']

In [0]:
display(df_level_6.head(10))

In [0]:
display(df_level_3.head(15))

In [0]:
def find_folder_ref_and_level(folder_name, hierarchy_df):
    for level in range(1, 8):
        name_col = f"Level_{level}_Name"
        ref_col = f"Level_{level}_Ref"
        if name_col in hierarchy_df.columns:
            match = hierarchy_df[hierarchy_df[name_col] == folder_name]
            if not match.empty:
                return {
                    "folder_ref": match.iloc[0]['folder_ref'],
                    "level": level,
                    "ref": match.iloc[0][ref_col]
                }
    return None


In [0]:
# Example usage:
result = find_folder_ref_and_level("Board, Committee, and Panel Members Details", hierarchy_df)
display(result)

In [0]:

# Build mapping using first occurrence of each folder_ref
folder_ref_to_first_name = {}

for _, row in hierarchy_df.iterrows():
    folder_ref = row['folder_ref']
    if folder_ref not in folder_ref_to_first_name:
        for level in range(1, 8):
            name_col = f"Level_{level}_Name"
            if name_col in row and pd.notnull(row[name_col]) and str(row[name_col]).strip():
                folder_ref_to_first_name[folder_ref] = row[name_col]
                break

# Display the mapping
mapping_df = pd.DataFrame(list(folder_ref_to_first_name.items()), columns=["folder_ref", "first_name"])
print(mapping_df)


In [0]:
len(mapping_df)

## Feature

In [0]:
import pandas as pd

excel_file = pd.ExcelFile("/Workspace/Users/hakeemfujah@tfl.gov.uk/Corporate Archives/Autoclassification Scheme/Split Series Content 07.01.2025.xlsx")
display(pd.DataFrame(excel_file.sheet_names, columns=["Sheet Names"]))

In [0]:
import openpyxl



# Function to check sheet tab colors in an Excel workbook
def check_sheet_tab_colors(file_path):
    # Load the workbook
    workbook = openpyxl.load_workbook(file_path)
    
    # Dictionary to store sheet names and their tab colors
    sheet_colors = {}

    # Iterate through each sheet in the workbook
    for sheet_name in workbook.sheetnames:
        sheet = workbook[sheet_name]
        tab_color = sheet.sheet_properties.tabColor
        
        # Check if tabColor is set
        if tab_color is not None:
            # Convert RGB color to hex format if available
            color_value = tab_color.rgb if tab_color.rgb else str(tab_color)
            sheet_colors[sheet_name] = color_value

    # Return the result
    return sheet_colors

# Example usage (replace with actual file path)
file_path = "/Workspace/Users/hakeemfujah@tfl.gov.uk/Corporate Archives/Autoclassification Scheme/Split Series Content 07.01.2025.xlsx"
colored_sheets = check_sheet_tab_colors(file_path)

# Print the results
if colored_sheets:
    print("The following sheets have tab colors:")
    for sheet, color in colored_sheets.items():
        print(f"- {sheet}: {color}")
else:
    print("No sheets in the workbook have tab colors.")

In [0]:
colored_sheets = pd.DataFrame(colored_sheets.items(), columns=["Sheet Name", "Tab Color"])
display(colored_sheets)

In [0]:
green_sheets = colored_sheets[colored_sheets["Tab Color"] == "FF92D050"]
display(green_sheets)

In [0]:
dfs = {}
for sheet_name in green_sheets["Sheet Name"]:
    df = pd.read_excel("/Workspace/Users/hakeemfujah@tfl.gov.uk/Corporate Archives/Autoclassification Scheme/Split Series Content 07.01.2025.xlsx", sheet_name=sheet_name)
    dfs[f"df_{sheet_name}"] = df
    display(df)

In [0]:
df.columns

In [0]:
display(df)

In [0]:
display(pd.DataFrame(df.columns, columns=["Column Names"]))

In [0]:
dfs.keys()

In [0]:
dfs['df_LT000072 - 57'].columns

In [0]:
for key in dfs.keys():
    print(f"Columns for {key}:")
    for col in dfs[key].columns:
        print(col)
    print()

In [0]:
for key in dfs.keys():
    if "New SysID" in dfs[key].columns:
        continue
    dfs[key].rename(columns={dfs[key].columns[0]: "New SysID"}, inplace=True)

In [0]:
dfs_int = {}
cols_of_interest = ["New SysID", "Title", "Date", "Description"] # reconsider including CatalogueStatus, df_LT000072 - 57 is missing that column 
for key in dfs.keys():
    dfs_int[key] = dfs[key][cols_of_interest]

In [0]:
missing_catalogue_status = [key for key in dfs.keys() if "CatalogueStatus" not in dfs[key].columns]
display(pd.DataFrame(missing_catalogue_status, columns=["Missing CatalogueStatus"]))

In [0]:
for key in dfs_int.keys():
    print(f"Columns for {key}:")
    for col in dfs_int[key].columns:
        print(col)
    print()

In [0]:
display(dfs_int['df_LT000115 - 251'])

In [0]:
for key in dfs_int.keys():
    print(f"Columns for {key}:")
    print(dfs_int[key].info())

In [0]:
full_df = pd.concat([df.astype(str) for df in dfs_int.values()])
display(full_df)

In [0]:
full_df.info()

In [0]:
full_df['New SysID'].unique()

In [0]:
full_df[full_df['New SysID'] == 'LT000258/011']

In [0]:
['Same as 210', 'Deaccessin - PHYSICALLY DONE', 'Delete on moving', 'Same as 188', 'Lent 1994, never returned', 'LT000259/001/001/007 - Lent 1998, never returned', 'Deaccessioned - PHYSICALLY DONE', 'Remove on mapping', 'Remove when mapping', 'Deacession - on moving', 'Deacession - PHYSICALLY DONE', 'Deaccession - pHYSICALLY DONE', 'Deaccession - PHYSICALLY DONE', 'Deaccession - on moving', 'Deaccession - PHYSICally done', 'nan', 'Never accessioned', 'Missing', 'DEACCESSION?']

In [0]:
filtered_df = full_df[full_df['New SysID'].str.contains('LT')]

In [0]:
len(filtered_df)

In [0]:
filtered_df.head(15)

In [0]:
folder_mapping.columns

In [0]:

# Create mapping dictionary
#folder_mapping['folder_name'] = folder_mapping.apply(get_deepest_name, axis=1)
folder_name_map = dict(zip(folder_mapping['Reference'], folder_mapping['Name']))

# Map folder names to filtered_df
filtered_df['folder_name'] = filtered_df['New SysID'].map(folder_name_map)

# # Display the updated filtered_df
# print(filtered_df)


In [0]:
vc = filtered_df['New SysID'].value_counts()
less_than_10 = vc[vc < 2].sum()
more_than_10 = vc[vc >= 2].sum()
total = vc.sum()
proportions = {
    'less_than_10': less_than_10 / total,
    'more_than_10': more_than_10 / total
}
display(proportions)

In [0]:
vc = filtered_df['New SysID'].value_counts()
ids_with_2_or_more = vc[vc >= 2].index
filtered_df_2plus = filtered_df[filtered_df['New SysID'].isin(ids_with_2_or_more)]
display(filtered_df_2plus)

In [0]:
filtered_df_2plus.isnull().sum()

In [0]:
# Example usage:
result = find_folder_ref_and_level("	Railway Board Meetings", hierarchy_df)
display(result)

In [0]:
folder_mapping[folder_mapping['Reference'] == 'LT000258/002/001']

In [0]:
df_excel['Ref No'].unique()

In [0]:
from difflib import get_close_matches

# Get all unique SysIDs in folder_mapping
valid_sysids = set(folder_mapping['Reference'])

# Find SysIDs in filtered_df that are not in folder_mapping
invalid_sysids = set(filtered_df['New SysID']) - valid_sysids

# For each invalid SysID, find the closest match in valid_sysids
similar_sysids = []
for sysid in invalid_sysids:
    matches = get_close_matches(sysid, valid_sysids, n=3, cutoff=0.7)
    similar_sysids.append({
        'invalid_sysid': sysid,
        'closest_matches': matches
    })

# Display as DataFrame
display(pd.DataFrame(similar_sysids))

In [0]:
display(filtered_df_2plus[filtered_df_2plus['folder_name'].isnull()])

In [0]:
poss_error = filtered_df_2plus[filtered_df_2plus['folder_name'].isnull()]['New SysID'].unique()
display(poss_error)

In [0]:
# For each SysID in poss_error, display its closest matches from similar_sysids
for i in poss_error:
    matches = next((item['closest_matches'] for item in similar_sysids if item['invalid_sysid'] == i), [])
    print(f"SysID: {i}\nClosest matches: {matches}\n")

In [0]:
X = filtered_df_2plus[filtered_df_2plus.notnull().all(axis=1)]

In [0]:
from sklearn.model_selection import StratifiedShuffleSplit

# Ensure all classes in train are also in test by using StratifiedShuffleSplit and filtering
sss = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
for train_idx, test_idx in sss.split(X, X['New SysID']):
    train_df = X.iloc[train_idx]
    test_df = X.iloc[test_idx]

# Remove any classes from train that are not present in test
test_classes = set(test_df['folder_name'])
train_df = train_df[train_df['folder_name'].isin(test_classes)]

In [0]:
display(train_df)
display(test_df)

In [0]:
# Check class distribution in train and test sets
train_counts = train_df['New SysID'].value_counts()
test_counts = test_df['New SysID'].value_counts()

# Combine into a DataFrame for comparison
import pandas as pd
class_representation = pd.DataFrame({
    'train_count': train_counts,
    'test_count': test_counts
}).fillna(0).astype(int)

display(class_representation)

In [0]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder

# Combine Title and Description into a single feature
train_features = train_df['Title'].fillna('') + ' ' + train_df['Description'].fillna('')
test_features = test_df['Title'].fillna('') + ' ' + test_df['Description'].fillna('')

# Vectorize the combined text features
vectorizer = TfidfVectorizer()
X_train = vectorizer.fit_transform(train_features)
X_test = vectorizer.transform(test_features)

# Encode the target variable
le = LabelEncoder()
y_train = le.fit_transform(train_df['folder_name'])
y_test = le.transform(test_df['folder_name'])

In [0]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score

# Train a simple classifier
clf = LogisticRegression(max_iter=1000)
clf.fit(X_train, y_train)

# Predict on test set
y_pred = clf.predict(X_test)

# Evaluate performance
report = classification_report(y_test, y_pred, target_names=le.classes_)
accuracy = accuracy_score(y_test, y_pred)

print("Accuracy:", accuracy)
print(report)

In [0]:
from sklearn.multiclass import OneVsRestClassifier
from sklearn.linear_model import LogisticRegression

# Train a OneVsRest classifier for multiclass prediction
ovr_clf = OneVsRestClassifier(LogisticRegression(max_iter=1000))
ovr_clf.fit(X_train, y_train)

# Predict on test set using the OneVsRest classifier
y_pred = ovr_clf.predict(X_test)
test_df['predicted_folder_name'] = le.inverse_transform(y_pred)


In [0]:
display(test_df[['Title', 'Description', 'folder_name', 'predicted_folder_name']])

In [0]:
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix

# Evaluate performance on test set
accuracy = accuracy_score(y_test, y_pred)
report = classification_report(y_test, y_pred, target_names=le.classes_)
conf_matrix = confusion_matrix(y_test, y_pred)

print("Accuracy:", accuracy)
print(report)
print("Confusion Matrix:\n", conf_matrix)

In [0]:
from transformers import pipeline

# Load summarization pipeline
summarizer = pipeline("summarization", model="facebook/bart-large-cnn")

def summarize_text(text, max_length=60, min_length=10):
    if not text or not isinstance(text, str) or text.strip() == "":
        return ""
    summary = summarizer(text, max_length=max_length, min_length=min_length, do_sample=False)
    return summary[0]['summary_text']

# Apply summarization to Description in train and test sets
train_df['Description_Summary'] = train_df['Description'].apply(summarize_text)
test_df['Description_Summary'] = test_df['Description'].apply(summarize_text)

# Combine Title and Description_Summary into a single feature
train_features = train_df['Title'].fillna('') + ' ' + train_df['Description_Summary'].fillna('')
test_features = test_df['Title'].fillna('') + ' ' + test_df['Description_Summary'].fillna('')

# Vectorize the combined text features
vectorizer = TfidfVectorizer()
X_train = vectorizer.fit_transform(train_features)
X_test = vectorizer.transform(test_features)

In [0]:
# Request user input for Title and Description
input_title = input("Enter the Title: ")
input_description = input("Enter the Description: ")

# Summarize the input description
input_description_summary = summarize_text(input_description)

# Combine Title and summarized Description
input_features = input_title + ' ' + input_description_summary

# Vectorize the input features
input_vector = vectorizer.transform([input_features])

# Predict the folder name using the trained OneVsRest classifier
predicted_label = ovr_clf.predict(input_vector)
predicted_folder_name = le.inverse_transform(predicted_label)[0]

print("Predicted folder name:", predicted_folder_name)

In [0]:
display(folder_mapping)

In [0]:
import numpy as np
import pandas as pd

# Get feature names from the vectorizer
feature_names = np.array(vectorizer.get_feature_names_out())

# For each class, get top 5 features by coefficient magnitude, excluding the label itself if present
rows = []
for idx, class_label in enumerate(le.classes_):
    class_coefs = ovr_clf.estimators_[idx].coef_.flatten()
    label_tokens = set(str(class_label).lower().split())
    mask = np.array([not any(token in fn for token in label_tokens) for fn in feature_names])
    filtered_coefs = class_coefs[mask]
    filtered_features = feature_names[mask]
    top_indices = np.argsort(np.abs(filtered_coefs))[-5:][::-1]
    top_features = filtered_features[top_indices]
    rows.append([class_label] + list(top_features))

top_features_df = pd.DataFrame(rows, columns=['Label'] + [f'Top_{i+1}' for i in range(5)])
display(top_features_df)

In [0]:
display(filtered_df)

In [0]:
pip install databricks_langchain